# 🎮 Phase 4: Collaborative Filtering & Matrix Factorization

> **Mục tiêu của Notebook:**
> 1. **Data Preparation:** Tải `interactions.parquet` (814,586 tương tác sạch) và thực hiện Train/Test Split.
> 2. **Matrix Factorization (SVD):** Huấn luyện mô hình phân rã ma trận thưa User-Item (TruncatedSVD, $k=64$ latent dimensions).
> 3. **Model Evaluation:** Đánh giá độ chính xác dự báo (RMSE, MAE).
> 4. **Top-K Recommendation:** Sinh danh sách gợi ý Top-10 game cho các User thực tế.
> 5. **User/Item Behavioral Clustering:** Phân cụm K-Means trên không gian latent factors để khám phá nhóm game có hành vi tương đồng.
> 6. **Model Artifact Export:** Lưu mô hình đã huấn luyện vào `models/collaborative/`.

In [ ]:
import os
import sys
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")
from src.models.collaborative.matrix_factorization import SVDRecommender
from src.models.collaborative.clustering import CollaborativeClusterer

# Thiết lập biểu đồ
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 120

print("[*] Libraries and modules imported successfully!")

## 1. Load Silver Interactions & Train/Test Split

Chia tập tương tác thành 80% Train và 20% Test theo cách ngẫu nhiên có cố định seed.

In [ ]:
interactions_path = "../data/silver/interactions.parquet"
items_path = "../data/silver/item_features.parquet"

df_interactions = pl.read_parquet(interactions_path)
df_items = pl.read_parquet(items_path)

print(f"[+] Total Clean Interactions: {len(df_interactions):,}")
print(f"[+] Total Items: {len(df_items):,}")

# Shuffle & Train/Test Split (80/20)
df_shuffled = df_interactions.sample(fraction=1.0, shuffle=True, seed=42)
split_idx = int(len(df_shuffled) * 0.8)
df_train = df_shuffled[:split_idx]
df_test = df_shuffled[split_idx:]

print(f"[+] Train size: {len(df_train):,} ({len(df_train)/len(df_interactions)*100:.1f}%)")
print(f"[+] Test size:  {len(df_test):,} ({len(df_test)/len(df_interactions)*100:.1f}%)")

## 2. Train TruncatedSVD Collaborative Filtering Model

Xây dựng ma trận thưa User-Item $R$ và phân rã thành User Factors $U$ và Item Factors $V$ ($k=64$ chiều ẩn).

In [ ]:
svd_model = SVDRecommender(n_factors=64, random_state=42)
R, stats = svd_model.fit_transform_interactions(df_train)

print("[+] SVD Training Completed!")
for k, v in stats.items():
    print(f"    - {k}: {v}")
print(f"    - User Factors Shape: {svd_model.user_factors.shape}")
print(f"    - Item Factors Shape: {svd_model.item_factors.shape}")

## 3. Evaluation on Test Interactions

Tính toán sai số RMSE (Root Mean Squared Error) và MAE (Mean Absolute Error) trên tập Test.

In [ ]:
eval_sample = df_test.sample(n=20000, seed=42)
metrics = svd_model.evaluate(eval_sample)

print("🎯 Test Evaluation Metrics:")
print(f"    - RMSE: {metrics['rmse']:.4f}")
print(f"    - MAE:  {metrics['mae']:.4f}")

## 4. Top-K Recommendation for Active Users

Lựa chọn một số user tích cực nhất để sinh gợi ý Top-5 game kèm thông tin tên game và thể loại.

In [ ]:
# Tìm user có nhiều tương tác nhất
top_users = df_interactions.group_by("user_id").len().sort("len", descending=True).head(3)["user_id"].to_list()
item_title_map = dict(zip(df_items["parent_asin"].to_list(), df_items["title"].to_list()))
item_cat_map = dict(zip(df_items["parent_asin"].to_list(), df_items["main_category"].to_list()))

for user_id in top_users:
    user_history = df_train.filter(pl.col("user_id") == user_id)["parent_asin"].to_list()
    recs = svd_model.recommend_top_k(user_id=user_id, user_interacted_items=user_history, top_k=5)
    
    print(f"👤 User: {user_id} (Đã chơi/đánh giá {len(user_history)} games)")
    print("=" * 85)
    print(f"{'Rank':<5} | {'Score':<8} | {'Parent ASIN':<15} | {'Category':<15} | {'Game Title'}")
    print("-" * 85)
    for rank, (asin, score) in enumerate(recs, 1):
        title = item_title_map.get(asin, "Unknown")
        cat = item_cat_map.get(asin, "Video Games")
        print(f"{rank:<5} | {score:<8.3f} | {asin:<15} | {str(cat)[:15]:<15} | {title}")
    print("\n")

## 5. Behavioral Clustering on Item Latent Factors (K-Means)

Phân cụm 25,612 game thành 8 cụm hành vi tương tác dựa trên Item Latent Vectors $V$.

In [ ]:
clusterer = CollaborativeClusterer(n_clusters=8, random_state=42)
item_ids = [svd_model.idx2item[i] for i in range(len(svd_model.item_factors))]
df_clusters = clusterer.fit_item_clusters(svd_model.item_factors, item_ids)

# Phân bố số lượng game trong từng cụm
cluster_counts = df_clusters.group_by("cf_cluster_id").len().sort("cf_cluster_id")
print("📊 Phân bố số lượng game theo Cụm hành vi (CF Clusters):")
display(cluster_counts.to_pandas())

# Vẽ biểu đồ phân bố cụm
plt.figure(figsize=(9, 5))
sns.barplot(x=cluster_counts["cf_cluster_id"].to_numpy(), y=cluster_counts["len"].to_numpy(), palette="viridis")
plt.title("Distribution of Games across 8 CF Behavioral Clusters", fontsize=13, fontweight='bold')
plt.xlabel("Cluster ID")
plt.ylabel("Number of Games")
plt.tight_layout()
plt.show()

## 6. Lưu Artifacts Mô hình

Lưu mô hình SVD và Clusterer đã huấn luyện để phục vụ cho Hybrid Recommender và API/Agent.

In [ ]:
os.makedirs("../models/collaborative", exist_ok=True)
svd_model.save_model("../models/collaborative")
clusterer.save("../models/collaborative/item_clusterer.joblib")
print("[+] Collaborative filtering model artifacts saved successfully!")